# Lesson 3: working with state spaces

This lesson can be downloaded as a notebook, a notebook for colab and a python file [here](https://marmote.gitlabpages.inria.fr/marmote/python_downloads.html)

In [1]:
import marmote.core as marmotecore
import marmote.markovchain as mmc
import numpy as np

State spaces for `Marmote`'s Markov chains are specified with `MarmoteSet` objects.
This lesson presents the principal features of this object, and the currently available implementations.

## Functionalities of MarmoteSet: example with intervals and boxes

Basic sets in `Marmote` are intervals and "boxes": the object classes are
`MarmoteInterval` and `MarmoteBox`.

Creation of an (integer) interval between 0 and 6 included.

In [2]:
itl = marmotecore.MarmoteInterval(0,6)

In [3]:
print( "name = ", itl.className())
print( "cardinal = ", itl.Cardinal())
print(itl)

name =  MarmoteInterval
cardinal =  7
Interval( 0..6 )


Creation of a 2-d box with size 2 in the first dimension and 3 in the second one.
Note that ranges will be 0..1 and 0..2.

In [4]:
box = marmotecore.MarmoteBox( [ 2, 3 ] )

Inspecting the propreties of the object

In [5]:
print( "name = ", box.className())
print( "cardinal = ", box.Cardinal())
print(box)

name =  MarmoteBox
cardinal =  6
Box( [ 0..1 ] x [ 0..2 ] )


Listing all the states

In [6]:
box.Enumerate()

'(   0,   0) (   0,   1) (   0,   2) (   1,   0) (   1,   1) (   1,   2) '

## States and their indices

Every state has an index and conversely. Indices range from 0 to Cardinal()-1.
The relevant methods are:

* `Index( state )` to get the index of some state
* `DecodeState( idx )` to get the state corresponding to some index

In [7]:
box.Index( [1,0] )

3

In [8]:
box.DecodeState( 4 )

array([1, 1])

### Printing states

When printing states, it is possible to get both its index information and its description.

`Marmote` has a "policy" for writing states which is controlled as follows. First we save the current policy
using the method `stateWritePolicy()`.

In [9]:
polsave = marmotecore.stateWritePolicy()

Next we manipulate the way states are printed with the method `setStateWritePolicy()`. We set in sequence the policy to its three possible values: `STATE_INDEX`, `STATE_BOTH` and `STATE_FULL`.

In [10]:
marmotecore.setStateWritePolicy( marmotecore.STATE_INDEX )
print( "Format with index: ", box.FormatState( 4 ) )
marmotecore.setStateWritePolicy( marmotecore.STATE_BOTH )
print( "Format with both: ", box.FormatState( 4 ) )
marmotecore.setStateWritePolicy( marmotecore.STATE_FULL )
print( "Format with full description: ", box.FormatState( 4 ) )

Format with index:  4 
Format with both:  4:{   1   1} 
Format with full description:     1   1 


Finally we restore the policy to what it was previously.

In [11]:
marmotecore.setStateWritePolicy(polsave)

### Checking membership

Membership is tested with the method `Belongs`.

In [12]:
print( itl.Belongs( np.array([4]) ) )    # 4 is in [0..6]
print( itl.Belongs( np.array([40]) ) )   # 40 is not
print( box.Belongs( np.array([4,4]) ) )  # a state outside the 2-d box
print( box.Belongs( np.array([1,1]) ) )  # a state inside

True
False
False
True


### Looping through the states

The easiest way to loop through all states is to use a `for` loop.
It enumerates the states in the form: (index, array representation).

In [13]:
for s in box:
    print(s)

(0, array([0, 0]))
(1, array([0, 1]))
(2, array([0, 2]))
(3, array([1, 0]))
(4, array([1, 1]))
(5, array([1, 2]))


There exist other ways to inspect `MarmoteSet` objects and make the correspondence between *states* and *indices*.
The following one uses the standard way to walk through a set using methods:

* `FirstState(state)` to initialize the state to the "zero state"
* `Index(state)` to convert a state to its index
* `NextState(state)` to move the state its follower in the set's order
* `IsFirst(state)` to test if the state is the "zero state".
  
All use a `state` variable which is an array of integers. Some methods *modify* this array.

In [14]:
marmotecore.setStateWritePolicy( marmotecore.STATE_BOTH )
bbuf = box.StateBuffer()
box.FirstState(bbuf)           # the buffer is modified!
print(bbuf)
isover = False
while not isover:
    idx = box.Index(bbuf)
    print( box.FormatState( box.Index(bbuf) ) )
    box.NextState(bbuf)        # the buffer is modified!
    isover = box.IsFirst(bbuf)

[0 0]
0:{   0   0} 
1:{   0   1} 
2:{   0   2} 
3:{   1   0} 
4:{   1   1} 
5:{   1   2} 


## Other sets

The complete hierarchy of `MarmoteSet` objects is currently:

* `EmptySet`
* `MarmoteInterval`
  + `Integers`
* `MarmoteBox`
* `BinarySequence`
* `BinarySimplex`
* `Simplex`

### Binary sequences

In [15]:
biseq = marmotecore.BinarySequence( 6 )

In [16]:
print( "name = ", biseq.className())
print( "cardinal = ", biseq.Cardinal())
biseq.Enumerate()

name =  BinarySequence
cardinal =  64


'( 0 0 0 0 0 0) ( 0 0 0 0 0 1) ( 0 0 0 0 1 0) ( 0 0 0 0 1 1) ( 0 0 0 1 0 0) ( 0 0 0 1 0 1) ( 0 0 0 1 1 0) ( 0 0 0 1 1 1) ( 0 0 1 0 0 0) ( 0 0 1 0 0 1) ( 0 0 1 0 1 0) ( 0 0 1 0 1 1) ( 0 0 1 1 0 0) ( 0 0 1 1 0 1) ( 0 0 1 1 1 0) ( 0 0 1 1 1 1) ( 0 1 0 0 0 0) ( 0 1 0 0 0 1) ( 0 1 0 0 1 0) ( 0 1 0 0 1 1) ( 0 1 0 1 0 0) ( 0 1 0 1 0 1) ( 0 1 0 1 1 0) ( 0 1 0 1 1 1) ( 0 1 1 0 0 0) ( 0 1 1 0 0 1) ( 0 1 1 0 1 0) ( 0 1 1 0 1 1) ( 0 1 1 1 0 0) ( 0 1 1 1 0 1) ( 0 1 1 1 1 0) ( 0 1 1 1 1 1) ( 1 0 0 0 0 0) ( 1 0 0 0 0 1) ( 1 0 0 0 1 0) ( 1 0 0 0 1 1) ( 1 0 0 1 0 0) ( 1 0 0 1 0 1) ( 1 0 0 1 1 0) ( 1 0 0 1 1 1) ( 1 0 1 0 0 0) ( 1 0 1 0 0 1) ( 1 0 1 0 1 0) ( 1 0 1 0 1 1) ( 1 0 1 1 0 0) ( 1 0 1 1 0 1) ( 1 0 1 1 1 0) ( 1 0 1 1 1 1) ( 1 1 0 0 0 0) ( 1 1 0 0 0 1) ( 1 1 0 0 1 0) ( 1 1 0 0 1 1) ( 1 1 0 1 0 0) ( 1 1 0 1 0 1) ( 1 1 0 1 1 0) ( 1 1 0 1 1 1) ( 1 1 1 0 0 0) ( 1 1 1 0 0 1) ( 1 1 1 0 1 0) ( 1 1 1 0 1 1) ( 1 1 1 1 0 0) ( 1 1 1 1 0 1) ( 1 1 1 1 1 0) ( 1 1 1 1 1 1) '

In [17]:
bbuf = biseq.StateBuffer()
biseq.FirstState(bbuf)
isover = False
while not isover:
    print( biseq.FormatState( biseq.Index(bbuf) ) )
    biseq.NextState(bbuf)
    isover = biseq.IsFirst(bbuf)

0:{0 0 0 0 0 0} 
1:{0 0 0 0 0 1} 
2:{0 0 0 0 1 0} 
3:{0 0 0 0 1 1} 
4:{0 0 0 1 0 0} 
5:{0 0 0 1 0 1} 
6:{0 0 0 1 1 0} 
7:{0 0 0 1 1 1} 
8:{0 0 1 0 0 0} 
9:{0 0 1 0 0 1} 
10:{0 0 1 0 1 0} 
11:{0 0 1 0 1 1} 
12:{0 0 1 1 0 0} 
13:{0 0 1 1 0 1} 
14:{0 0 1 1 1 0} 
15:{0 0 1 1 1 1} 
16:{0 1 0 0 0 0} 
17:{0 1 0 0 0 1} 
18:{0 1 0 0 1 0} 
19:{0 1 0 0 1 1} 
20:{0 1 0 1 0 0} 
21:{0 1 0 1 0 1} 
22:{0 1 0 1 1 0} 
23:{0 1 0 1 1 1} 
24:{0 1 1 0 0 0} 
25:{0 1 1 0 0 1} 
26:{0 1 1 0 1 0} 
27:{0 1 1 0 1 1} 
28:{0 1 1 1 0 0} 
29:{0 1 1 1 0 1} 
30:{0 1 1 1 1 0} 
31:{0 1 1 1 1 1} 
32:{1 0 0 0 0 0} 
33:{1 0 0 0 0 1} 
34:{1 0 0 0 1 0} 
35:{1 0 0 0 1 1} 
36:{1 0 0 1 0 0} 
37:{1 0 0 1 0 1} 
38:{1 0 0 1 1 0} 
39:{1 0 0 1 1 1} 
40:{1 0 1 0 0 0} 
41:{1 0 1 0 0 1} 
42:{1 0 1 0 1 0} 
43:{1 0 1 0 1 1} 
44:{1 0 1 1 0 0} 
45:{1 0 1 1 0 1} 
46:{1 0 1 1 1 0} 
47:{1 0 1 1 1 1} 
48:{1 1 0 0 0 0} 
49:{1 1 0 0 0 1} 
50:{1 1 0 0 1 0} 
51:{1 1 0 0 1 1} 
52:{1 1 0 1 0 0} 
53:{1 1 0 1 0 1} 
54:{1 1 0 1 1 0} 
55:{1 1 0 1 1 1} 
56

Illustration of membership tests. When a state does not belong to the set, its index is 0 by convention and a warning is issued.

In [18]:
st1 = np.array( [1, 0, 1, 1, 0, 1], dtype=int )
print( biseq.Belongs( st1 ) )
print( biseq.Index( st1 ) )

True
45


In [19]:
st2 = np.array( [2, 4, 1, 1, 0, 1], dtype=int )
print( biseq.Belongs( st2 ) )
print( biseq.Index( st2 ) )

False
0


### The set of integers

`Marmote` has also a set for all integers!

In [20]:
itg = marmotecore.MarmoteIntegers()

In [21]:
print( "name = ", itg.className())
print( "cardinal = ", itg.Cardinal()) ## -2 is the convention for 'infinite cardinal'
print( itg.IsFinite() )

name =  MarmoteIntegers
cardinal =  -2
False


Membership tests

In [22]:
print( itg.Belongs( [4] ) )
print( itg.Belongs( [-3] ) )

True
False


In [23]:
print( itg.Index( [-3] ) )

0


### Simplices

Simplex sets are sometimes difficult to enumerate. Not with `Marmote`!

In [24]:
splx = marmotecore.Simplex( 7, 3 )

In [25]:
splx.Enumerate()

'( 0 0 0 0 0 0 3) ( 0 0 0 0 0 1 2) ( 0 0 0 0 0 2 1) ( 0 0 0 0 0 3 0) ( 0 0 0 0 1 0 2) ( 0 0 0 0 1 1 1) ( 0 0 0 0 1 2 0) ( 0 0 0 0 2 0 1) ( 0 0 0 0 2 1 0) ( 0 0 0 0 3 0 0) ( 0 0 0 1 0 0 2) ( 0 0 0 1 0 1 1) ( 0 0 0 1 0 2 0) ( 0 0 0 1 1 0 1) ( 0 0 0 1 1 1 0) ( 0 0 0 1 2 0 0) ( 0 0 0 2 0 0 1) ( 0 0 0 2 0 1 0) ( 0 0 0 2 1 0 0) ( 0 0 0 3 0 0 0) ( 0 0 1 0 0 0 2) ( 0 0 1 0 0 1 1) ( 0 0 1 0 0 2 0) ( 0 0 1 0 1 0 1) ( 0 0 1 0 1 1 0) ( 0 0 1 0 2 0 0) ( 0 0 1 1 0 0 1) ( 0 0 1 1 0 1 0) ( 0 0 1 1 1 0 0) ( 0 0 1 2 0 0 0) ( 0 0 2 0 0 0 1) ( 0 0 2 0 0 1 0) ( 0 0 2 0 1 0 0) ( 0 0 2 1 0 0 0) ( 0 0 3 0 0 0 0) ( 0 1 0 0 0 0 2) ( 0 1 0 0 0 1 1) ( 0 1 0 0 0 2 0) ( 0 1 0 0 1 0 1) ( 0 1 0 0 1 1 0) ( 0 1 0 0 2 0 0) ( 0 1 0 1 0 0 1) ( 0 1 0 1 0 1 0) ( 0 1 0 1 1 0 0) ( 0 1 0 2 0 0 0) ( 0 1 1 0 0 0 1) ( 0 1 1 0 0 1 0) ( 0 1 1 0 1 0 0) ( 0 1 1 1 0 0 0) ( 0 1 2 0 0 0 0) ( 0 2 0 0 0 0 1) ( 0 2 0 0 0 1 0) ( 0 2 0 0 1 0 0) ( 0 2 0 1 0 0 0) ( 0 2 1 0 0 0 0) ( 0 3 0 0 0 0 0) ( 1 0 0 0 0 0 2) ( 1 0 0 0 0 1 1) ( 1 0 0 0 0 2

In [26]:
print( splx.Belongs( [ 0, 0, 0, 0, 0, 0 ] ) )
print( splx.Belongs( [ 0, 0, 0, 0, 0, 0, 3 ] ) )
print( splx.Belongs( [ 0, -1, 1, 0, 0, 0, 3 ] ) )

False
True
False


In [27]:
bsplx = marmotecore.BinarySimplex( 7, 3 )

In [28]:
bsplx.Enumerate()

'( 1 1 1 0 0 0 0) ( 1 1 0 1 0 0 0) ( 1 1 0 0 1 0 0) ( 1 1 0 0 0 1 0) ( 1 1 0 0 0 0 1) ( 1 0 1 1 0 0 0) ( 1 0 1 0 1 0 0) ( 1 0 1 0 0 1 0) ( 1 0 1 0 0 0 1) ( 1 0 0 1 1 0 0) ( 1 0 0 1 0 1 0) ( 1 0 0 1 0 0 1) ( 1 0 0 0 1 1 0) ( 1 0 0 0 1 0 1) ( 1 0 0 0 0 1 1) ( 0 1 1 1 0 0 0) ( 0 1 1 0 1 0 0) ( 0 1 1 0 0 1 0) ( 0 1 1 0 0 0 1) ( 0 1 0 1 1 0 0) ( 0 1 0 1 0 1 0) ( 0 1 0 1 0 0 1) ( 0 1 0 0 1 1 0) ( 0 1 0 0 1 0 1) ( 0 1 0 0 0 1 1) ( 0 0 1 1 1 0 0) ( 0 0 1 1 0 1 0) ( 0 0 1 1 0 0 1) ( 0 0 1 0 1 1 0) ( 0 0 1 0 1 0 1) ( 0 0 1 0 0 1 1) ( 0 0 0 1 1 1 0) ( 0 0 0 1 1 0 1) ( 0 0 0 1 0 1 1) ( 0 0 0 0 1 1 1) '

More set manipulations to be seen in App_Lesson1!